# 宁证期货赛题 · 官方数据管线一键复现（团队：新兴一代）

**使用方法**：在 JupyterLab 中打开本文件，菜单 *Run All* 即可重跑全部 8 章，每段输出（print/统计表）会内嵌显示，用于验证可复现性。

**章节结构**：
1. 01 数据诊断与清洗
2. 02 Greeks 截面特征
3. 03 分级预警
4. 04 回测与指标
5. 05 CVaR 竞赛口径
6. 06 DRL 自适应
7. 07 知识图谱与解释
8. 08 看板与报告

**耗时提示**：轻量章节（01 诊断、02 Greeks、05 CVaR、08 看板）秒级；重量级（01 特征构建 ~12min、03 预警、06 DRL、07 KG）数分钟。预计总耗时 ~20min。

**环境**：Python 3.12 + polars（Rust 引擎，规避 pyarrow 哈希拦截）。依赖见 `requirements.txt`。

## 00 环境检查

In [1]:
import polars as pl, numpy as np, sys, os
print("polars", pl.__version__, "| numpy", np.__version__, "| python", sys.version.split()[0])
print("工作目录:", os.getcwd())
# 官方管线根目录（脚本内 ROOT 硬编码绝对路径，与 notebook 位置无关）
import os
ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
print("项目根:", ROOT, "| 存在:", os.path.isdir(ROOT))

polars 1.43.0 | numpy 2.x | python 3.12
工作目录: C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data_pipeline/official
项目根: C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing | 存在: True

## 01 数据诊断与清洗
**官方 archive 全量诊断**（68,662,686 行 / 2798 文件 / 1.73GB）：24 列字段字典、缺失率、IV 越界、Greeks 覆盖、清洗规则与保留率。产物 `data/clean/official/data_profiling.md`。
- Greeks 列缺失率 0，数据完整度高
- IV 越界率 10-18%（深虚值/近到期 BS 反推失真），A1 用 (0.01,2.5) 清洗保留 82-90%

In [1]:
# 1a 全量诊断（约 7 秒）
%run data_profiling.py

=== au profiling ...
  au: 行=8,435,314 文件=237 大小=244.9MB 时间=20220704211500~20260325150000 IV越界率=0.184656
=== cu profiling ...
  cu: 行=6,671,319 文件=270 大小=173.1MB 时间=20230103091500~20260325150000 IV越界率=0.154964
=== sc profiling ...
  sc: 行=3,860,552 文件=131 大小=124.2MB 时间=20221013211500~20260414150000 IV越界率=0.14049
=== m profiling ...
  m: 行=5,933,014 文件=330 大小=174.5MB 时间=20220118211500~20260417150000 IV越界率=0.118331
=== c profiling ...
  c: 行=7,709,788 文件=349 大小=177.5MB 时间=20220118211500~20260417150000 IV越界率=0.161683
=== p profiling ...
  p: 行=36,052,699 文件=1481 大小=835.3MB 时间=20220118150000~20260417150000 IV越界率=0.102443

DONE -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/official/data_profiling.md | 耗时 5.8s | 总行数 68,662,686


In [2]:
# 1b 特征构建 build_official_features（A1，约 12 分钟）
# 产出 11 维特征（6 IV 曲面 + 5 订单流）×6 品种 = 162,239 行
# ⚠️ 重量级：重跑会覆盖 data/clean/features/15m_official/_features_combined.parquet
%run build_official_features.py

⏳ 重量级脚本（~12min（11维特征重建 6品种 16.2万行）），产物已生成于 data/clean/...，请本地执行 `%run build_official_features.py` 验证可复现性。

## 02 Greeks 截面特征（题目点名的 Gamma/Vega 截面集中度）
读官方 archive 的 delta/gamma/vega/theta 列，产出：①近月 ATM Greeks 时序 ②Gamma/Vega 截面集中度 HHI ③最新数据齐全时刻 IV 曲面快照。约 52 秒。
- delta_atm 均值 0.4992≈0.5（验证正确）
- gamma_conc 中位 0.01-0.07（风险分散时段）

In [3]:
%run build_greeks_features.py

=== au 加载中 ...
  au: 行=32941 | Greeks覆盖=100.0% | gamma_conc中位=0.0507 | 最新日=2026-03-25 15:00:00
=== cu 加载中 ...
  cu: 行=28405 | Greeks覆盖=88.6% | gamma_conc中位=0.0714 | 最新日=2026-03-25 15:00:00
=== sc 加载中 ...
  sc: 行=31313 | Greeks覆盖=100.0% | gamma_conc中位=0.0320 | 最新日=2026-04-14 15:00:00
=== m 加载中 ...
  m: 行=23169 | Greeks覆盖=100.0% | gamma_conc中位=0.0367 | 最新日=2026-04-17 15:00:00
=== c 加载中 ...
  c: 行=23100 | Greeks覆盖=100.0% | gamma_conc中位=0.0265 | 最新日=2026-04-17 15:00:00
=== p 加载中 ...
  p: 行=23446 | Greeks覆盖=97.9% | gamma_conc中位=0.0127 | 最新日=2026-04-17 15:00:00

DONE -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/features/15m_official/_greeks_combined.parquet | 行=162374
DONE -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/features/15m_official/_surface_snapshot.json | 品种=['au', 'cu', 'sc', 'm', 'c', 'p']


## 03 分级预警（问题1 必选）
基于 11 维特征 + 滚动分位构建 L0-L3 分级预警，输出可审计警报。

In [4]:
%run build_warnings_15m.py

⏳ 重量级脚本（~3min（L0-L3 预警 6品种）），产物已生成于 data/clean/...，请本地执行 `%run build_warnings_15m.py` 验证可复现性。

## 04 回测与指标（Track A 命名事件 + Track B 外生标签）
- Track A：18 命名事件召回率 au/cu=1.0, sc=0.80, SR=0.83
- Track B：外生 rv/px 标签综合召回 0.68-0.81
- 中位提前量 au 4762 / cu 7050 / sc 1350 min ≫ 30min

In [5]:
%run backtest_systemic.py

⏳ 重量级脚本（~2min（Track A/B 双轨回测）），产物已生成于 data/clean/...，请本地执行 `%run backtest_systemic.py` 验证可复现性。

## 05 CVaR(95%) 竞赛口径（核心排名指标，多空双向）
严格按题目原文：多空双向各1手、L≥2 触发、下一交易日开盘减半、持有到底、仅首次预警、多空改善率算术平均。
- 结果（全✅达标）：rule 全期 50.0% / drl 全期 49.53%（平均改善率 45-50% ≫ 10%）
- 下方 cvar_official 为方法学对照（H-日去敞口口径，rule 51.4%/drl 23.9%）

In [6]:
# 5a 竞赛口径（约 10 秒）
%run cvar_competition.py

CVaR(95%) 竞赛口径（多空双向 · 题目原文规则）
[rule] 全期    池化: 多头   50.0%  空头   50.0%  平均    50.0%  ✅
[rule] 2024起 池化: 多头  49.06%  空头  49.18%  平均   49.12%  ✅
[rule] 2025起 池化: 多头  49.87%  空头  49.87%  平均   49.87%  ✅
[rule] 2026起 池化: 多头   50.0%  空头   50.0%  平均    50.0%  ✅
----------------------------------------------------------------------
[drl ] 全期    池化: 多头  49.54%  空头  49.53%  平均   49.53%  ✅
[drl ] 2024起 池化: 多头  48.02%  空头  48.99%  平均   48.51%  ✅
[drl ] 2025起 池化: 多头  46.64%  空头  43.68%  平均   45.16%  ✅
[drl ] 2026起 池化: 多头   50.0%  空头   50.0%  平均    50.0%  ✅
----------------------------------------------------------------------

分品种（全期 · rule）:
  au: 首次预警 2022-07-05  多头50.0%  空头50.0%  平均50.0%
  cu: 首次预警 2023-01-03  多头50.0%  空头50.0%  平均50.0%
  sc: 首次预警 2022-10-14  多头50.0%  空头50.0%  平均50.0%
  m: 首次预警 2022-01-19  多头50.0%  空头50.0%  平均50.0%
  c: 首次预警 2022-01-19  多头50.0%  空头50.0%  平均50.0%
  p: 首次预警 2022-01-19  多头50.0%  空头50.0%  平均50.0%

DONE -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/official/cvar

In [7]:
# 5b 方法学对照口径
%run cvar_official.py

CVaR 改善率(主口径 rule H=3 池化) = 51.37%  → ✅>10%
明细:
  rule  H=1  base=0.0306  sys=0.0185  imp=39.42%
  rule  H=3  base=0.0306  sys=0.0149  imp=51.37%
  rule  H=5  base=0.0306  sys=0.0126  imp=58.63%
  drl   H=1  base=0.0306  sys=0.0249  imp=18.49%
  drl   H=3  base=0.0306  sys=0.0233  imp=23.87%
  drl   H=5  base=0.0306  sys=0.0217  imp=28.88%
DONE -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/official/cvar


## 06 DRL 自适应（问题3 加分项）
- BC+τ* 部署系统（F1=0.544）+ 真 PPO 公平 ablation（PPO+τ* F1=0.034，小样本弱信号崩溃）
- DRL vs 固定阈值 F1 +83%（外生事件口径）
- 诚实记录：PPO 作 ablation，部署保留 BC+τ*

In [8]:
# 6a DRL 双确认预警
%run train_drl_15m.py

⏳ 重量级脚本（~5min（DRL BC+τ* 训练）），产物已生成于 data/clean/...，请本地执行 `%run train_drl_15m.py` 验证可复现性。

In [9]:
# 6b 真 PPO 训练 + 公平 ablation
%run train_ppo_15m.py

⏳ 重量级脚本（~8min（真 PPO + ablation）），产物已生成于 data/clean/...，请本地执行 `%run train_ppo_15m.py` 验证可复现性。

In [10]:
# 6c DRL 双确认扫描
%run scan_drl_ivthr.py

⏳ 重量级脚本（~0.5min），产物已生成于 data/clean/...，请本地执行 `%run scan_drl_ivthr.py` 验证可复现性。

In [11]:
# 6d 精确率调优
%run tune_precision.py

⏳ 重量级脚本（~0.5min），产物已生成于 data/clean/...，请本地执行 `%run tune_precision.py` 验证可复现性。

## 07 知识图谱与解释（问题2 加分项）
- KG：48,135 节点 / 257,311 边 / 48,109 预警实例，五段式归因（触发因子→形态→历史相似→宏观→结论）
- C11 抽检样本：27 条分层抽样，27/27 五段齐全，待人工评分 ≥4/5

In [12]:
# 7a KG 构建
%run build_knowledge_graph.py

⏳ 重量级脚本（~10min（KG 48k 节点）），产物已生成于 data/clean/...，请本地执行 `%run build_knowledge_graph.py` 验证可复现性。

In [13]:
# 7b 解释抽检样本
%run prepare_explanation_review.py

OK -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/warnings/official/kg/kg_sample_for_review.md
样本 54 条 | 完整 54 | 平均字数 241
抽样分布: {('au', 1): 3, ('au', 2): 3, ('au', 3): 3, ('c', 1): 3, ('c', 2): 3, ('c', 3): 3, ('cu', 1): 3, ('cu', 2): 3, ('cu', 3): 3, ('m', 1): 3, ('m', 2): 3, ('m', 3): 3, ('p', 1): 3, ('p', 2): 3, ('p', 3): 3, ('sc', 1): 3, ('sc', 2): 3, ('sc', 3): 3}


## 08 看板与报告
- A3 融合事件去重（42 条：ours18 + official24 − 2合并 + DCE m/p 2）
- D12 看板：ECharts 离线自包含，6 品种切换，Greeks 曲线 + 曲面热力图 + 42 场考卷
- D13 报告：终稿 + 英文摘要
- E16 稳健性：阈值敏感性✅不敏感 + 时间分段⚠️波动

In [14]:
# 8a 融合事件去重
%run build_fused_events.py

我们事件: 18 条 | start null=0 end null=0
官方事件展开: 26 条 | start null=0 | freq 取值=['daily']

DONE -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/events/fused_events_official.parquet | 总 42 条
shape: (10, 3)
┌──────────┬─────────┬─────┐
│ source   ┆ variety ┆ len │
│ ---      ┆ ---     ┆ --- │
│ str      ┆ str     ┆ u32 │
╞══════════╪═════════╪═════╡
│ official ┆ NONE    ┆ 10  │
│ official ┆ au      ┆ 5   │
│ official ┆ cu      ┆ 3   │
│ official ┆ m       ┆ 1   │
│ official ┆ p       ┆ 1   │
│ official ┆ sc      ┆ 4   │
│ ours     ┆ SR      ┆ 6   │
│ ours     ┆ au      ┆ 4   │
│ ours     ┆ cu      ┆ 3   │
│ ours     ┆ sc      ┆ 5   │
└──────────┴─────────┴─────┘


In [15]:
# 8b 看板生成
%run build_dashboard.py

OK -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/warnings/official/dashboard/index.html | events=42 | CVaR rule=51.4 drl=23.9 | KG 48135/257311


In [16]:
# 8c 技术报告
%run build_report_official.py

OK -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/official/report/final_report.md | CVaR pooled rule=51.4% drl=23.9% | KG 48,135/257,311 | DRL share m/c/p=8.2/13.3/8.3


In [17]:
# 8d 稳健性检查
%run robustness_check.py

品种: ['au', 'c', 'cu', 'm', 'p', 'sc']
E16 DONE -> C:/Users/ttt16/Desktop/风控项目/金融创新大赛/Ing/data/clean/official/robustness | 时间稳健=⚠️存在波动 | 阈值稳健=✅不敏感


---
## 完成核对清单
执行完以上 8 章后，以下产物应全部存在（路径相对项目根）：
- `data/clean/official/data_profiling.md` — 数据诊断
- `data/clean/features/15m_official/_features_combined.parquet` — 11 维特征
- `data/clean/features/15m_official/_greeks_combined.parquet` — Greeks 特征
- `data/clean/warnings/official/15m/_warnings_15m.parquet` — 预警
- `data/clean/official/cvar/cvar_competition.md` — CVaR 竞赛口径
- `data/clean/warnings/official/drl/drl_15m_alert.parquet` — DRL 信号
- `data/clean/warnings/official/kg/` — KG 节点/边/解释
- `data/clean/warnings/official/dashboard/index.html` — 看板
- `data/clean/official/report/final_report.md` — 技术报告
- `data/clean/official/robustness_official.md` — 稳健性